# QC check: RR monthly-total consistency (DuckDB/Parquet)

This notebook runs and validates the first QC check:

1. Keep only specifiers with an exact rank-1 RR match.
2. Recompute each monthly total from consensus daily values (median across members per day).
3. Compare to matched RR monthly value.
4. Flag all days in that file+month as pass if |diff| <= tolerance, else fail.

QC outputs are written to Parquet under qc_root:
- qc_sessions
- daily_qc_results
- daily_qc_status

In [1]:
# Setup roots and imports for parquet QC.

import os
from pathlib import Path

import duckdb

from src.rainfall_rescue_sqlite.parquet_ingest import (
    default_ensemble_parquet_root,
    default_rainfall_rescue_parquet_root,
 )
from src.rainfall_rescue_sqlite.parquet_qc_exact_monthly import (
    default_qc_parquet_root,
    run_exact_monthly_consistency_check_parquet,
 )
from src.rainfall_rescue_sqlite.parquet_similarity import default_comparison_parquet_root

pdir = Path(os.environ["PDIR"])
ensemble_dataset_root = default_ensemble_parquet_root()
comparison_root = default_comparison_parquet_root()
rr_dataset_root = default_rainfall_rescue_parquet_root()
qc_root = default_qc_parquet_root()

print(f"ensemble dataset: {ensemble_dataset_root}")
print(f"comparison root:  {comparison_root}")
print(f"RR dataset:       {rr_dataset_root}")
print(f"QC root:          {qc_root}")

ensemble dataset: /data/scratch/philip.brohan/ADRQ/ensemble_transcriptions_parquet
comparison root:  /data/scratch/philip.brohan/ADRQ/monthly_similarity_parquet
RR dataset:       /data/scratch/philip.brohan/ADRQ/Rainfall-Rescue/rainfall_rescue_parquet
QC root:          /data/scratch/philip.brohan/ADRQ/qc_parquet


In [2]:
# Smoke test on a small file-id slice.

result_smoke = run_exact_monthly_consistency_check_parquet(
    ensemble_dataset_root=ensemble_dataset_root,
    comparison_root=comparison_root,
    rr_dataset_root=rr_dataset_root,
    qc_root=qc_root,
    tolerance=0.01,
    start_file_id=1,
    end_file_id=200,
)

result_smoke

ExactMonthlyQCResult(qc_session_id=1, files_processed=200, day_rows_written=74400, pass_rows=16957, fail_rows=57443, exact_files_seen=116, non_exact_files_seen=84)

In [ ]:
# Optional full run on local machine (small tests only).

run_full = False

if run_full:
    result_full = run_exact_monthly_consistency_check_parquet(
        ensemble_dataset_root=ensemble_dataset_root,
        comparison_root=comparison_root,
        rr_dataset_root=rr_dataset_root,
        qc_root=qc_root,
        tolerance=0.01,
    )
    print(result_full)
else:
    print("run_full is False - set to True to run the full QC check.")

## Running at scale on SLURM

The full dataset has ~514,000 file IDs.  At ~10 file IDs/second the
single-threaded run above would take ~14 hours.

Instead, submit the distributed pipeline (array + merge) from the cluster
login node with the cell below.  100 shards of ~5,140 file IDs each take
~9 minutes per shard; the merge runs once after all shards succeed.

```
scripts/slurm/submit_qc.sh
```

The cells below show how to check progress and inspect results after the
merge job completes.


In [3]:
# Get the current max file_id so QC_TOTAL_FILE_IDS is set correctly.

conn = duckdb.connect()
try:
    max_file_id = conn.execute(
        f"SELECT MAX(file_id) FROM read_parquet('{ensemble_dataset_root / 'ensemble_files' / '*.parquet'}')"
    ).fetchone()[0]
finally:
    conn.close()

print(f"Max file_id: {max_file_id}")
print()
print("Submit the SLURM pipeline with:")
print(f"  QC_TOTAL_FILE_IDS={max_file_id} scripts/slurm/submit_qc.sh")
print()
print("Or with custom shard count (e.g. 200 shards for ~7 min each):")
print(f"  QC_NUM_SHARDS=200 QC_TOTAL_FILE_IDS={max_file_id} scripts/slurm/submit_qc.sh")

Max file_id: 584513

Submit the SLURM pipeline with:
  QC_TOTAL_FILE_IDS=584513 scripts/slurm/submit_qc.sh

Or with custom shard count (e.g. 200 shards for ~7 min each):
  QC_NUM_SHARDS=200 QC_TOTAL_FILE_IDS=584513 scripts/slurm/submit_qc.sh


In [4]:
# Show recent QC sessions from parquet outputs.

conn = duckdb.connect()
try:
    rows = conn.execute(
        f"""
        SELECT qc_session_id, started_at, completed_at, status, promotion_threshold, message
        FROM read_parquet('{qc_root / 'qc_sessions' / '*.parquet'}')
        ORDER BY qc_session_id DESC
        LIMIT 10
        """
    ).fetchall()
finally:
    conn.close()

for r in rows:
    print(
        f"session={r[0]:>4}  status={r[3]:<24}  "
        f"started={r[1]}  completed={r[2]}"
    )
    if r[5]:
        print(f"  message: {r[5]}")

session=   2  status=success                   started=2026-07-23T19:58:45+00:00  completed=2026-07-23T20:02:12+00:00
  message: exact_monthly_consistency merged 100 shards, 217438836 rows (40023077 pass, 177415759 fail)
session=   1  status=success                   started=2026-07-23T19:48:11+00:00  completed=2026-07-23T19:48:25+00:00
  message: exact_monthly_consistency wrote 74400 day rows (16957 pass, 57443 fail); final status (16957 pass, 57443 fail)


In [5]:
# Summarise pass/fail totals for the latest QC session.

conn = duckdb.connect()
try:
    latest = conn.execute(
        f"SELECT MAX(qc_session_id) FROM read_parquet('{qc_root / 'qc_sessions' / '*.parquet'}')"
    ).fetchone()[0]

    if latest is None:
        raise RuntimeError("No QC sessions found. Run a QC cell first.")

    print(f"Latest QC session: {latest}")

    check_counts = conn.execute(
        f"""
        SELECT qc_flag, COUNT(*) AS n
        FROM read_parquet('{qc_root / 'daily_qc_results' / '*.parquet'}')
        WHERE qc_session_id = ?
          AND check_name = 'exact_monthly_consistency'
        GROUP BY qc_flag
        ORDER BY qc_flag
        """
        , [latest]
    ).fetchall()

    print("Per-check row counts:")
    for flag, n in check_counts:
        print(f"  {flag}: {n}")

    final_counts = conn.execute(
        f"""
        SELECT final_flag, COUNT(*) AS n
        FROM read_parquet('{qc_root / 'daily_qc_status' / '*.parquet'}')
        WHERE qc_session_id = ?
        GROUP BY final_flag
        ORDER BY final_flag
        """
        , [latest]
    ).fetchall()
finally:
    conn.close()

print("Final day-level status counts:")
for flag, n in final_counts:
    print(f"  {flag}: {n}")

Latest QC session: 2
Per-check row counts:
  fail: 177415759
  pass: 40023077
Final day-level status counts:
  fail: 177415759
  pass: 40023077


In [6]:
# Inspect one specifier and month-by-month outcomes from the latest QC session.

specifier = "DRain_1891-1900_RainNos_Glamorgan-3"

conn = duckdb.connect()
try:
    latest = conn.execute(
        f"SELECT MAX(qc_session_id) FROM read_parquet('{qc_root / 'qc_sessions' / '*.parquet'}')"
    ).fetchone()[0]

    file_row = conn.execute(
        f"""
        SELECT file_id, file_name
        FROM read_parquet('{ensemble_dataset_root / 'ensemble_files' / '*.parquet'}')
        WHERE file_name = ?
        LIMIT 1
        """
        , [f"{specifier}.json"]
    ).fetchone()

    if file_row is None:
        raise RuntimeError(f"Specifier not found: {specifier}")

    print(f"file_id={file_row[0]}  file_name={file_row[1]}")

    rows = conn.execute(
        f"""
        SELECT month, day_of_month, qc_flag, qc_score, details_json
        FROM read_parquet('{qc_root / 'daily_qc_results' / '*.parquet'}')
        WHERE qc_session_id = ?
          AND file_id = ?
          AND check_name = 'exact_monthly_consistency'
        ORDER BY month, day_of_month
        LIMIT 120
        """
        , [latest, file_row[0]]
    ).fetchall()
finally:
    conn.close()

for row in rows:
    month = row[0]
    day_of_month = row[1]
    qc_flag = row[2]
    qc_score = row[3]
    print(
        f"month={month:>2} day={day_of_month:>2}  "
        f"flag={qc_flag:<5} score={qc_score}"
    )

file_id=549733  file_name=DRain_1891-1900_RainNos_Glamorgan-3.json
month= 1 day= 1  flag=fail  score=0.0
month= 1 day= 2  flag=fail  score=0.0
month= 1 day= 3  flag=fail  score=0.0
month= 1 day= 4  flag=fail  score=0.0
month= 1 day= 5  flag=fail  score=0.0
month= 1 day= 6  flag=fail  score=0.0
month= 1 day= 7  flag=fail  score=0.0
month= 1 day= 8  flag=fail  score=0.0
month= 1 day= 9  flag=fail  score=0.0
month= 1 day=10  flag=fail  score=0.0
month= 1 day=11  flag=fail  score=0.0
month= 1 day=12  flag=fail  score=0.0
month= 1 day=13  flag=fail  score=0.0
month= 1 day=14  flag=fail  score=0.0
month= 1 day=15  flag=fail  score=0.0
month= 1 day=16  flag=fail  score=0.0
month= 1 day=17  flag=fail  score=0.0
month= 1 day=18  flag=fail  score=0.0
month= 1 day=19  flag=fail  score=0.0
month= 1 day=20  flag=fail  score=0.0
month= 1 day=21  flag=fail  score=0.0
month= 1 day=22  flag=fail  score=0.0
month= 1 day=23  flag=fail  score=0.0
month= 1 day=24  flag=fail  score=0.0
month= 1 day=25  flag

## Interactive QC flag map

This mirrors the interactive map workflow in the metadata-matching notebook,
but colours stations by QC flag for a selected date and QC session.

For the current monthly-consistency check output, flags are pass/fail.

Clicking a station copies its specifier to the clipboard.

In [8]:
# Demonstrate the interactive daily QC map, rendered inline.

import importlib.util
import sys
from datetime import date
from pathlib import Path

from IPython.display import display
import src.rainfall_rescue_sqlite as _pkg

repo_root = Path(_pkg.__file__).resolve().parents[2]
qc_map_script_path = repo_root / "scripts" / "diagnostics" / "plot_daily_qc_interactive.py"

spec_qc_map = importlib.util.spec_from_file_location("daily_qc_map_plot", qc_map_script_path)
if spec_qc_map is None or spec_qc_map.loader is None:
    raise ImportError(f"Unable to load module from {qc_map_script_path}")

daily_qc_map_plot = importlib.util.module_from_spec(spec_qc_map)
sys.modules[spec_qc_map.name] = daily_qc_map_plot
spec_qc_map.loader.exec_module(daily_qc_map_plot)

target_date = date(1891, 11, 13)
qc_session_id = None
qc_map_output_path = Path(
    f"{os.getenv('PDIR')}/diagnostics/daily_qc_map_{target_date.isoformat()}.html"
 )

interactive_qc_fig = daily_qc_map_plot.build_figure(
    target_date=target_date,
    ensemble_db=None,
    qc_session_id=qc_session_id,
    output_path=qc_map_output_path,
    backend="duckdb",
    ensemble_dataset_root=ensemble_dataset_root,
    comparison_root=comparison_root,
    qc_root=qc_root,
)

display(interactive_qc_fig)
print(f"Saved interactive HTML with click-to-copy UI: {qc_map_output_path}")

Saved interactive HTML with click-to-copy UI: /data/scratch/philip.brohan/ADRQ/diagnostics/daily_qc_map_1891-11-13.html
